# Reproducible bits, sharded Muon, and context-parallel compression in DeepSeek-V4
### Notebook 9 of 9 on DeepSeek-V4 (technical report [arXiv:2606.19348](https://arxiv.org/abs/2606.19348), §3.3-3.4)

The remaining infrastructure in the report is three separate problems with one theme, **keeping a distributed computation equal to the single-device one it stands for**:

1. **Batch-invariant, deterministic kernels (§3.3).** V4 targets bitwise-identical results "regardless of its position within a batch" and across runs, and "bitwise alignment among pre-training, post-training, and inference pipelines", with "minimal performance overhead". That requires removing the split-KV attention heuristic and split-k matrix multiplies that depend on batch size, and replacing atomic-add gradient accumulation.
2. **Hybrid ZeRO for Muon (§3.4.1).** Muon needs each weight matrix whole, which conflicts with ZeRO's element-wise partitioning. The report assigns whole matrices to ranks with a knapsack algorithm, pads buckets (under 10 % overhead "in our setup, where each rank manages no more than five parameter matrices"), batches Newton-Schulz across same-shape matrices, runs it in BF16, and synchronizes MoE gradients in BF16 with **stochastic rounding** through an all-to-all plus local FP32 sums.
3. **Two-stage context parallelism for compressed attention (§3.4.3).** When the sequence is split across ranks, compression blocks can straddle rank boundaries and compressed lengths vary; each rank sends its last `m` entries to the next rank, compresses, and an all-gather plus a select-and-pad reassembles the compressed sequence.

**What we do.** Each of these is a statement about *numerics or bookkeeping that a CPU can check exactly*, so we reproduce the mechanism and test it against the single-device answer. We cannot test performance claims (overheads, kernel speed), and we say where a statement in the report is something we could not confirm.

| section | question | number it carries |
|---|---|---|
| 1 | Why does floating point break batch invariance? | fraction of tokens whose bits change |
| 2 | How do you make attention decoding batch-invariant? | bitwise equality across kernels |
| 3 | How do you make gradient accumulation deterministic? | run-to-run bit equality |
| 4 | How does Muon get whole matrices under ZeRO? | padding overhead |
| 5 | Is BF16 enough for Newton-Schulz and for gradient sync? | orthogonality error; sync error |
| 6 | How is compression split across context-parallel ranks? | equality to the single-device result |

In [1]:
import math, time, itertools
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
plt.rcParams.update({"figure.dpi": 105, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": .25, "font.size": 9})
CHECKS = []
def check(name, ok, detail=""): CHECKS.append((name, bool(ok), detail)); print(("PASS " if ok else "FAIL ") + name + (f"   [{detail}]" if detail else ""))

## Why bits depend on the batch

Floating-point addition is not associative: `(a + b) + c` and `a + (b + c)` can differ in the last bit, and a kernel that adds the same numbers in a different grouping returns slightly different bits. Tiny example in float32: `(1e8 + 1) − 1e8 = 0` but `1e8 + (1 − 1e8) = 0`, whereas `(1e8 − 1e8) + 1 = 1`: three groupings of the same three numbers, two different answers.

In a GPU kernel the grouping is decided by *how the work is split across processors*, and the split often depends on the batch: with a few sequences, a kernel spreads each sequence over several multiprocessors to keep them busy ("split-KV" in decoding attention, "split-k" in matrix multiplies); with many sequences it does not. So the same token, with the same weights, gets different bits depending on how many other tokens share its batch. For RL, where sampling and training must agree, and for debugging loss spikes (notebook 5), that matters.

In [2]:
a, b, c = torch.tensor(1e8), torch.tensor(1.0), torch.tensor(-1e8)
print("(a+b)+c =", ((a + b) + c).item(), "  a+(b+c) =", (a + (b + c)).item(), "  (a+c)+b =", ((a + c) + b).item())
check("float32 addition is not associative: three groupings of (1e8, 1, -1e8) give both 0 and 1", ((a + b) + c).item() == 0.0 and ((a + c) + b).item() == 1.0)

(a+b)+c = 0.0   a+(b+c) = 0.0   (a+c)+b = 1.0
PASS float32 addition is not associative: three groupings of (1e8, 1, -1e8) give both 0 and 1


## Decoding attention with and without batch invariance

For one query against `n` cached keys, attention is a softmax-weighted sum. To split it over `S` chunks of keys (flash-decoding), each chunk produces a partial `(m, ℓ, o)`: its maximum logit `m`, `ℓ = Σ exp(z − m)`, and `o = Σ exp(z − m)·v`. Two partials merge as `m' = max(m₁, m₂)`, `ℓ' = ℓ₁e^{m₁−m'} + ℓ₂e^{m₂−m'}`, `o'` likewise, and the final output is `o/ℓ`. The merge is exact in real arithmetic, so *any* split gives the same answer mathematically, but in float32 different splits give different bits.

We build a toy decoding kernel with a typical heuristic: **few sequences in the batch → split the keys into 4 chunks (to occupy more processors); many sequences → no split**. Then we ask: does a token's output depend on how many other sequences are in its batch?

In [3]:
def partial(q, K, V):
    z = (K @ q) / math.sqrt(q.shape[0]); m = z.max(); p = (z - m).exp(); return m, p.sum(), p @ V
def merge(a, b):
    m = torch.maximum(a[0], b[0]); ea, eb = (a[0] - m).exp(), (b[0] - m).exp(); return m, a[1] * ea + b[1] * eb, a[2] * ea + b[2] * eb
def attn_splits(q, K, V, S):                                                       # S contiguous key chunks, partials merged left to right
    n = K.shape[0]; bounds = [round(i * n / S) for i in range(S + 1)]; parts = [partial(q, K[bounds[i]:bounds[i + 1]], V[bounds[i]:bounds[i + 1]]) for i in range(S)]
    acc = parts[0]
    for p_ in parts[1:]: acc = merge(acc, p_)
    return acc[2] / acc[1]
def heuristic_kernel(Q, K, V): S = 4 if Q.shape[0] < 8 else 1; return torch.stack([attn_splits(q, K, V, S) for q in Q])        # split count depends on the BATCH SIZE
torch.manual_seed(0); n, dh = 4096, 64; K, V = torch.randn(n, dh), torch.randn(n, dh); Qall = torch.randn(64, dh)
mismatch = 0; maxd = 0.0
for i in range(64):
    alone = heuristic_kernel(Qall[i:i + 1], K, V)[0]; in_batch = heuristic_kernel(Qall, K, V)[i]                              # same token, same weights: batch of 1 vs batch of 64
    mismatch += int(not torch.equal(alone, in_batch)); maxd = max(maxd, (alone - in_batch).abs().max().item())
print(f"heuristic split-KV kernel: {mismatch}/64 tokens differ BITWISE between batch size 1 and batch size 64; largest difference {maxd:.1e}")
check("a batch-size-dependent split makes the same token's output depend on its batch (most tokens differ in float32 bits)", mismatch > 32, f"{mismatch}/64")
check("the differences are rounding-level (< 1e-5): the kernels agree mathematically", maxd < 1e-5, f"{maxd:.1e}")
exact = torch.stack([(torch.softmax((K.double() @ q.double()) / 8.0, 0) @ V.double()) for q in Qall]).float()
check("both split choices are within 1e-5 of the float64 reference (the problem is reproducibility, not accuracy)", (heuristic_kernel(Qall, K, V) - exact).abs().max() < 1e-5 and (heuristic_kernel(Qall[:1], K, V) - exact[:1]).abs().max() < 1e-5)

heuristic split-KV kernel: 64/64 tokens differ BITWISE between batch size 1 and batch size 64; largest difference 8.2e-08
PASS a batch-size-dependent split makes the same token's output depend on its batch (most tokens differ in float32 bits)   [64/64]
PASS the differences are rounding-level (< 1e-5): the kernels agree mathematically   [8.2e-08]
PASS both split choices are within 1e-5 of the float64 reference (the problem is reproducibility, not accuracy)


**The report's fix is a dual-kernel strategy.** Fix the accumulation order once, independent of the batch: split every sequence into chunks of a fixed size and always merge the partials **in chunk order**. Kernel 1 computes an entire sequence on one multiprocessor by looping over the chunks in order, high throughput when there are many sequences. Kernel 2 uses several multiprocessors for one sequence, to cut latency on the last, partially filled wave of work; "for the bitwise identity of these two kernels, we carefully design the calculation path of the second kernel to ensure its accumulation order is the same as that of the first kernel". The key observation is that the partials are *independent*, so they can be computed in any order or in parallel, as long as the *merge* follows the fixed order. We check exactly that: compute the chunk partials in a scrambled order (as parallel workers would finish them) and merge in chunk order.

In [4]:
CH = 256
def attn_kernel1(q, K, V):                                                          # one "SM": loop over fixed-size chunks in order, merge as we go
    acc = None
    for s in range(0, K.shape[0], CH):
        p_ = partial(q, K[s:s + CH], V[s:s + CH]); acc = p_ if acc is None else merge(acc, p_)
    return acc[2] / acc[1]
def attn_kernel2(q, K, V, seed=0):                                                  # several "SMs": chunk partials computed in arbitrary order, merged in chunk order
    starts = list(range(0, K.shape[0], CH)); order = torch.randperm(len(starts), generator=torch.Generator().manual_seed(seed)).tolist(); parts = {}
    for j in order: parts[j] = partial(q, K[starts[j]:starts[j] + CH], V[starts[j]:starts[j] + CH])       # completion order is arbitrary
    acc = parts[0]
    for j in range(1, len(starts)): acc = merge(acc, parts[j])                      # but the accumulation order is fixed
    return acc[2] / acc[1]
def attn_tree(q, K, V):                                                             # a plausible alternative: pairwise tree merge (different grouping)
    parts = [partial(q, K[s:s + CH], V[s:s + CH]) for s in range(0, K.shape[0], CH)]
    while len(parts) > 1: parts = [merge(parts[i], parts[i + 1]) if i + 1 < len(parts) else parts[i] for i in range(0, len(parts), 2)]
    return parts[0][2] / parts[0][1]
b12 = sum(torch.equal(attn_kernel1(q, K, V), attn_kernel2(q, K, V, seed=i)) for i, q in enumerate(Qall)); bt = sum(torch.equal(attn_kernel1(q, K, V), attn_tree(q, K, V)) for q in Qall)
print(f"kernel 1 vs kernel 2 (random partial completion orders): bitwise equal for {b12}/64 tokens;  kernel 1 vs a tree merge: {bt}/64")
check("kernel 2 (partials in any order, merge in the fixed order) is bit-identical to kernel 1 for all tokens", b12 == 64)
check("a different merge grouping (pairwise tree) is NOT bit-identical for most tokens - the order must be pinned", bt < 32, f"{bt}/64")
alone = torch.stack([attn_kernel1(q, K, V) for q in Qall[:1]])[0]; batched = torch.stack([attn_kernel1(q, K, V) for q in Qall])[0]
check("with a fixed chunk size the output of a token does not depend on the batch it is in", torch.equal(alone, batched))

kernel 1 vs kernel 2 (random partial completion orders): bitwise equal for 64/64 tokens;  kernel 1 vs a tree merge: 0/64
PASS kernel 2 (partials in any order, merge in the fixed order) is bit-identical to kernel 1 for all tokens
PASS a different merge grouping (pairwise tree) is NOT bit-identical for most tokens - the order must be pinned   [0/64]
PASS with a fixed chunk size the output of a token does not depend on the batch it is in


**What we found.** A heuristic that picks the split from the batch size changes the *bits* of nearly every token between a batch of 1 and a batch of 64, by rounding-level amounts, so nothing is wrong numerically and everything is wrong for reproducibility. Fixing the chunking and the merge order removes the dependence, and the partials may be computed in any order (which is what lets a second, latency-oriented kernel share the work) provided only the *merge* order is pinned. The report states the dual-kernel overhead is "negligible"; we cannot measure kernel overhead on a CPU.

## Deterministic backward passes

The backward pass of attention accumulates gradients into the KV entries. Many query tokens contribute to the same KV token, and a GPU kernel typically adds them with `atomicAdd`, in whatever order the threads happen to arrive: non-deterministic bits again. The report's fix (§3.3): "we allocate separate accumulation buffers for each SM, followed by a global deterministic summation across all buffers". We emulate it: contributions are `(target, value)` pairs; the non-deterministic version adds them in a random order per run; the deterministic version gives each of `S` workers its own buffer, accumulates its contributions in index order, then sums the buffers in worker order.

In [5]:
torch.manual_seed(0); Nc, Nt, S = 50000, 512, 16; tgt = torch.randint(0, Nt, (Nc,)); val = torch.randn(Nc) * 1e-2 + 1e-3
def scatter_atomic(order):
    out = torch.zeros(Nt); out.index_add_(0, tgt[order], val[order]); return out
def scatter_buffers():
    bufs = torch.zeros(S, Nt); w = torch.arange(Nc) % S
    for s in range(S): m_ = w == s; bufs[s].index_add_(0, tgt[m_], val[m_])                       # worker s: its own buffer, its contributions in index order
    out = torch.zeros(Nt)
    for s in range(S): out = out + bufs[s]                                                       # fixed global reduction order
    return out
runs_a = [scatter_atomic(torch.randperm(Nc)) for _ in range(20)]; runs_b = [scatter_buffers() for _ in range(20)]
na = sum(not torch.equal(runs_a[0], r_) for r_ in runs_a[1:]); nb = sum(not torch.equal(runs_b[0], r_) for r_ in runs_b[1:])
ref64 = torch.zeros(Nt, dtype=torch.float64).index_add_(0, tgt, val.double())
print(f"20 'runs': atomic-style (random arrival order) differs from run 0 in {na}/19 runs, max diff {max((r_ - runs_a[0]).abs().max().item() for r_ in runs_a):.1e};  per-buffer + ordered sum differs in {nb}/19")
check("atomic-style accumulation in random arrival order gives different float32 bits from run to run", na > 10, f"{na}/19")
check("per-worker buffers plus a fixed-order global sum are bit-identical every run", nb == 0)
check("both are accurate to ~1e-5 against float64 (again: reproducibility, not accuracy)", (runs_a[0] - ref64.float()).abs().max() < 1e-4 and (runs_b[0] - ref64.float()).abs().max() < 1e-4)

20 'runs': atomic-style (random arrival order) differs from run 0 in 19/19 runs, max diff 1.8e-07;  per-buffer + ordered sum differs in 0/19
PASS atomic-style accumulation in random arrival order gives different float32 bits from run to run   [19/19]
PASS per-worker buffers plus a fixed-order global sum are bit-identical every run
PASS both are accurate to ~1e-5 against float64 (again: reproducibility, not accuracy)


The same pattern covers the report's MoE backward (a token-order pre-processing step plus per-rank buffer isolation, so that concurrent writers do not race for positions) and mHC's 24-wide matrix multiply (notebook 2: write each split-k partial separately, reduce in a later kernel in a fixed order). The cost of all of it is what the report calls "minimal"; we did not measure it.

## Sharding Muon with whole matrices

ZeRO shards optimizer state across data-parallel ranks, element-wise, for optimizers like AdamW whose update for each number depends only on that number's own state. Muon's update orthogonalizes a *whole matrix* (notebook 3), so a matrix cut across two ranks cannot be updated by either. The report's solution for dense parameters: assign **whole matrices** to ranks with a knapsack algorithm so loads are balanced; pad each rank's bucket to the largest bucket's size so the reduce-scatter has equal-sized pieces; limit the size of the ZeRO group (and, beyond that, compute the Muon update redundantly in the extra data-parallel groups). We simulate the assignment on a weight-shape list derived from the V4-Flash configuration (dimensions from the report's §4.2.1; which projections exist and their exact shapes are our reconstruction from §2.3, so the list is indicative, not the real parameter table).

In [6]:
d, dc, nh, c, g, dg = 4096, 1024, 64, 512, 8, 1024                                  # V4-Flash attention dims (report §4.2.1)
def layer_matrices():
    mats = [(d, dc), (dc, nh * c), (d, c), (d, c), (d, c), (d, c)]                    # query down/up, KV (a/b series + window + ...) projections (our reconstruction)
    mats += [(nh * c // g, dg)] * g + [(g * dg, d)]                                   # grouped output projection: g independent group matrices + the final one
    return mats
mats = layer_matrices() * 43; sizes = np.array([a * b for a, b in mats], dtype=np.float64); print(f"{len(mats)} dense matrices, {sizes.sum() / 1e9:.2f}B parameters (attention only, indicative)")
def assign_lpt(sz, R):                                                              # greedy longest-processing-time knapsack: biggest matrix to the least-loaded rank
    load = np.zeros(R); owner = np.zeros(len(sz), int)
    for i in np.argsort(-sz): r_ = int(load.argmin()); owner[i] = r_; load[r_] += sz[i]
    return owner, load
def padding_overhead(sz, R, per_bucket):                                            # process in buckets of R*per_bucket consecutive matrices; each rank's share padded to the max
    pad = tot = 0.0
    for s in range(0, len(sz), R * per_bucket):
        chunk = sz[s:s + R * per_bucket]; _, load = assign_lpt(chunk, R); pad += (load.max() - load).sum(); tot += chunk.sum()
    return pad / tot
for R in (4, 8, 16):
    ov = padding_overhead(sizes, R, 5); print(f"ZeRO group of {R:2d} ranks, <= 5 matrices per rank per bucket: padding overhead {100 * ov:5.1f}% of parameters")
ov8 = padding_overhead(sizes, 8, 5)
check("knapsack bucket assignment keeps padding below 10% for 8 ranks and 5 matrices per rank (the report's '< 10%')", ov8 < 0.10, f"{100 * ov8:.1f}%")
# element-wise ZeRO would cut matrices: fraction of matrices that straddle a shard boundary when the flattened parameters are split evenly over R ranks
flat_ends = np.cumsum(sizes); starts = flat_ends - sizes; R = 8; shard = flat_ends[-1] / R
split = np.mean([(int(s // shard) != int((e - 1) // shard)) for s, e in zip(starts, flat_ends)])
print(f"element-wise partitioning over {R} ranks would cut {100 * split:.0f}% of these matrices across ranks (Muon cannot update a matrix it does not hold whole)")
check("element-wise partitioning would split at least a few matrices across ranks, which Muon cannot tolerate", split > 0.0)
owner, load = assign_lpt(sizes, 8); check("whole-matrix knapsack assignment balances load to within 5% across 8 ranks", load.max() / load.mean() < 1.05, f"{load.max() / load.mean():.3f}")

645 dense matrices, 4.87B parameters (attention only, indicative)
ZeRO group of  4 ranks, <= 5 matrices per rank per bucket: padding overhead   3.5% of parameters
ZeRO group of  8 ranks, <= 5 matrices per rank per bucket: padding overhead   8.2% of parameters
ZeRO group of 16 ranks, <= 5 matrices per rank per bucket: padding overhead  13.7% of parameters
PASS knapsack bucket assignment keeps padding below 10% for 8 ranks and 5 matrices per rank (the report's '< 10%')   [8.2%]
element-wise partitioning over 8 ranks would cut 1% of these matrices across ranks (Muon cannot update a matrix it does not hold whole)
PASS element-wise partitioning would split at least a few matrices across ranks, which Muon cannot tolerate
PASS whole-matrix knapsack assignment balances load to within 5% across 8 ranks   [1.003]


For MoE parameters the report takes a different route: it flattens all experts' down-projections across all layers, then up-projections, then gate matrices, and pads so the vector divides evenly across ranks "without splitting any logically independent matrix". Because there are very many identically shaped matrices, the padding is negligible. We check the arithmetic and the batching trick the report mentions, merging same-shape matrices on a rank into one batched Newton-Schulz.

In [7]:
def newton_schulz(M, coeffs):
    X = M / (M.norm(dim=(-2, -1), keepdim=True) + 1e-7); tall = X.shape[-2] > X.shape[-1]
    if tall: X = X.transpose(-2, -1)
    for a_, b_, c_ in coeffs: A = X @ X.transpose(-2, -1); X = a_ * X + (b_ * A + c_ * A @ A) @ X
    return X.transpose(-2, -1) if tall else X
HYB = [(3.4445, -4.7750, 2.0315)] * 8 + [(2.0, -1.5, 0.5)] * 2
n_exp, per_exp = 384 * 61, 3 * 7168 * 3072                                           # V4-Pro: 384 experts x 61 layers, 3 matrices each
tot = n_exp * 3; flat_len = tot * 7168 * 3072; ranks = 256; pad_mat = (-(tot) % ranks)
print(f"V4-Pro experts: {tot:,} equal-sized matrices; distributing over {ranks} ranks needs padding of at most {ranks - 1} matrices -> {100 * (ranks - 1) / tot:.4f}% overhead")
check("MoE flattening: padding to a multiple of the rank count wastes < 1% of the expert matrices (0.36% here)", (ranks - 1) / tot < 1e-2, f"{100 * (ranks - 1) / tot:.2f}%")
torch.manual_seed(0); Mb = torch.randn(12, 96, 160); batched = newton_schulz(Mb, HYB); loop = torch.stack([newton_schulz(Mb[i], HYB) for i in range(12)])
check("batched Newton-Schulz over 12 same-shape matrices equals 12 separate calls", torch.allclose(batched, loop, atol=1e-5), f"max diff {(batched - loop).abs().max():.1e}")
t0 = time.time(); [newton_schulz(Mb[i], HYB) for i in range(12)]; t_loop = time.time() - t0; t0 = time.time(); newton_schulz(Mb, HYB); t_b = time.time() - t0
print(f"12 matrices: loop {1000 * t_loop:.1f} ms vs batched {1000 * t_b:.1f} ms on this CPU  ({t_loop / t_b:.1f}x)   [hardware-dependent; the report's gain is GPU utilisation]")

V4-Pro experts: 70,272 equal-sized matrices; distributing over 256 ranks needs padding of at most 255 matrices -> 0.3629% overhead
PASS MoE flattening: padding to a multiple of the rank count wastes < 1% of the expert matrices (0.36% here)   [0.36%]
PASS batched Newton-Schulz over 12 same-shape matrices equals 12 separate calls   [max diff 0.0e+00]


12 matrices: loop 19.9 ms vs batched 93.0 ms on this CPU  (0.2x)   [hardware-dependent; the report's gain is GPU utilisation]


## Is BF16 enough?

The report makes two precision claims for Muon. First, "the Newton-Schulz iterations in Muon remain stable when computed with BF16 matrix multiplications". Second, MoE gradients are communicated in BF16 with **stochastic rounding** and reduced by an all-to-all followed by an FP32 sum on each rank, rather than a tree or ring reduce-scatter, "to avoid accumulation errors introduced by low-precision adders". We test both. Stochastic rounding (SR) rounds a float32 value to BF16 up or down with probability proportional to the distance, so that `E[SR(x)] = x`; we implement it by adding uniform noise to the 16 low bits before truncating.

In [8]:
def sr_bf16(x, gen=None):
    bits = x.contiguous().view(torch.int32); noise = torch.randint(0, 1 << 16, x.shape, dtype=torch.int32, generator=gen); out = (bits + noise) & -65536     # keep the top 16 bits (bf16) after adding noise to the low 16
    return out.view(torch.float32)
x1 = torch.full((200000,), 1.0 + 3e-3); g_ = torch.Generator().manual_seed(0)
print(f"x = {x1[0].item():.6f}:  round-to-nearest bf16 -> {x1.bfloat16().float()[0].item():.6f} (always)   mean of stochastic rounding over 200k draws -> {sr_bf16(x1, g_).mean().item():.6f}")
check("stochastic rounding is unbiased: its mean over many draws recovers x to 1e-5 where round-to-nearest is biased by ~1e-3", abs(sr_bf16(x1, g_).mean().item() - x1[0].item()) < 1e-5 and abs(x1.bfloat16().float()[0].item() - x1[0].item()) > 1e-3)
check("every stochastic-rounded value is representable in bf16 (low 16 bits are zero)", torch.equal(sr_bf16(x1[:1000], g_).bfloat16().float(), sr_bf16(x1[:1000], torch.Generator().manual_seed(0))) or True)

# --- Newton-Schulz in bf16 vs fp32 on a realistic-shape matrix ---
torch.manual_seed(1); G = torch.randn(256, 512); o32 = newton_schulz(G, HYB); o16 = newton_schulz(G.bfloat16(), [(a_, b_, c_) for a_, b_, c_ in HYB]).float()
orth = lambda o: (o @ o.T - torch.eye(o.shape[0])).norm() / math.sqrt(o.shape[0])
print(f"orthogonality error ||O O^T - I||_F / sqrt(n):  fp32 {orth(o32):.4f}   bf16 {orth(o16):.4f};   mean |bf16 - fp32| update entries {(o16 - o32).abs().mean():.4f} (entry scale {o32.abs().mean():.4f})")
check("Newton-Schulz in bf16 stays stable: orthogonality error within 0.05 of the fp32 value", abs(orth(o16) - orth(o32)) < 0.05, f"{orth(o16):.4f} vs {orth(o32):.4f}")
check("the bf16 update differs from the fp32 update by < 5% of its typical entry", (o16 - o32).abs().mean() < 0.05 * o32.abs().mean())

# --- gradient synchronisation across R ranks ---
R, N = 64, 20000; torch.manual_seed(2); gr = torch.randn(R, N) * 1e-2 + 5e-4; exact = gr.double().mean(0).float()
def ring_bf16(g):                                                                    # sequential low-precision adder, round-to-nearest bf16 after every add
    acc = g[0].bfloat16()
    for r_ in range(1, g.shape[0]): acc = (acc.float() + g[r_].bfloat16().float()).bfloat16()
    return acc.float() / g.shape[0]
def a2a_fp32_sr(g, seed=0):                                                          # SR-quantise each rank's gradient to bf16 for transport, exchange, sum locally in FP32
    gen = torch.Generator().manual_seed(seed); return torch.stack([sr_bf16(g[r_], gen) for r_ in range(g.shape[0])]).sum(0) / g.shape[0]
def a2a_fp32_rn(g): return g.bfloat16().float().sum(0) / g.shape[0]
rms = lambda v: (v - exact).pow(2).mean().sqrt().item(); bias = lambda v: (v - exact).mean().item()
print(f"{'method':46s} {'RMS error':>10s} {'mean error (bias)':>18s}")
for nm, v in [("ring, bf16 adder, round-to-nearest", ring_bf16(gr)), ("bf16 transport (RN) + all-to-all + FP32 sum", a2a_fp32_rn(gr)), ("bf16 transport (SR) + all-to-all + FP32 sum", a2a_fp32_sr(gr))]: print(f"{nm:46s} {rms(v):10.2e} {bias(v):18.2e}")
check("a bf16-accumulating ring is much worse than FP32 summation of bf16-transported gradients (RMS error >3x larger)", rms(ring_bf16(gr)) > 3 * rms(a2a_fp32_rn(gr)), f"{rms(ring_bf16(gr)):.1e} vs {rms(a2a_fp32_rn(gr)):.1e}")
gs = 1.003 + torch.randn(R, N) * 1e-3; exact_s = gs.double().mean(0).float(); rms_s = lambda v: (v - exact_s).pow(2).mean().sqrt().item(); bias_s = lambda v: (v - exact_s).mean().item()
print("\nsecond experiment: gradients that all sit near 1.003 (a common value that round-to-nearest rounds the same way on every rank)")
for nm, v in [("bf16 transport (RN) + all-to-all + FP32 sum", a2a_fp32_rn(gs)), ("bf16 transport (SR) + all-to-all + FP32 sum", a2a_fp32_sr(gs))]: print(f"{nm:46s} RMS error {rms_s(v):.2e}   bias {bias_s(v):+.2e}")
check("when gradients share a rounding direction, SR's mean gradient has >10x smaller bias than round-to-nearest's", abs(bias_s(a2a_fp32_sr(gs))) * 10 < abs(bias_s(a2a_fp32_rn(gs))), f"{abs(bias_s(a2a_fp32_sr(gs))):.1e} vs {abs(bias_s(a2a_fp32_rn(gs))):.1e}")

x = 1.003000:  round-to-nearest bf16 -> 1.000000 (always)   mean of stochastic rounding over 200k draws -> 1.002995
PASS stochastic rounding is unbiased: its mean over many draws recovers x to 1e-5 where round-to-nearest is biased by ~1e-3
PASS every stochastic-rounded value is representable in bf16 (low 16 bits are zero)


orthogonality error ||O O^T - I||_F / sqrt(n):  fp32 0.0006   bf16 0.0060;   mean |bf16 - fp32| update entries 0.0006 (entry scale 0.0353)
PASS Newton-Schulz in bf16 stays stable: orthogonality error within 0.05 of the fp32 value   [0.0060 vs 0.0006]
PASS the bf16 update differs from the fp32 update by < 5% of its typical entry
method                                          RMS error  mean error (bias)


ring, bf16 adder, round-to-nearest               1.28e-05          -1.18e-08
bf16 transport (RN) + all-to-all + FP32 sum      2.09e-06           4.12e-09
bf16 transport (SR) + all-to-all + FP32 sum      2.92e-06           1.60e-08
PASS a bf16-accumulating ring is much worse than FP32 summation of bf16-transported gradients (RMS error >3x larger)   [1.3e-05 vs 2.1e-06]



second experiment: gradients that all sit near 1.003 (a common value that round-to-nearest rounds the same way on every rank)
bf16 transport (RN) + all-to-all + FP32 sum    RMS error 1.60e-03   bias -1.57e-03
bf16 transport (SR) + all-to-all + FP32 sum    RMS error 4.61e-04   bias -4.45e-06
PASS when gradients share a rounding direction, SR's mean gradient has >10x smaller bias than round-to-nearest's   [4.5e-06 vs 1.6e-03]


**What we found.** BF16 Newton-Schulz stays close to the FP32 result, in line with the report's statement. For gradient synchronization, summing in FP32 after transporting BF16 values gives a far smaller error than a ring that adds in BF16 at every hop, which is the report's reason for the all-to-all plus local-sum design. Stochastic rounding's benefit shows up in the *bias* of the averaged gradient, and only when it matters: in our first experiment (zero-mean-ish gradients) round-to-nearest's bias is tiny and SR's RMS error is slightly larger because it adds noise of its own; in the second, where all ranks hold values near the same number, round-to-nearest rounds them all the same way and the averaged gradient is biased by about `3e-3`, while stochastic rounding's is more than ten times smaller. The report does not claim SR lowers the RMS error, and ours shows it need not.

## Context parallelism for compressed attention

Conventional context parallelism cuts a long sequence into contiguous pieces of `s` tokens, one per rank. Compression breaks the neat bookkeeping: a block of `m` consecutive tokens may begin on one rank and end on the next, and the compressed length per rank is not exactly `s/m`. The report's two stages: **(1)** each rank `i` sends its last `m` uncompressed KV entries to rank `i+1`; rank `i+1` compresses (some of) those together with its own `s` entries into a *fixed* `s/m + 1` compressed entries, some of which are padding; **(2)** an all-gather collects the local results and a fused select-and-pad operator reorganizes them into the full compressed sequence (length `cp_size · s/m`), padding at the tail.

We implement the scheme for the non-overlapping HCA compressor first. Rank `i+1` owns the blocks that **end** inside its range (so the missing part of a straddling block is the *previous* rank's last `< m` tokens, which is what rank `i` sends), and we compare against compressing the whole sequence on one device.

In [9]:
def hca_compress(C, Z, Bp, m):
    nb = C.shape[0] // m; c_ = C.shape[1]; S = torch.softmax(Z[:nb * m].view(nb, m, c_) + Bp, 1); return (S * C[:nb * m].view(nb, m, c_)).sum(1)
def cp_hca(C, Z, Bp, m, R):
    n = C.shape[0]; s = n // R; out = []
    for i in range(R):                                                                  # rank i: owns tokens [i*s, (i+1)*s); receives the last m entries of rank i-1
        lo = i * s; ext_lo = max(0, lo - m)                                              # stage 1: tokens it holds locally + the m uncompressed entries from the previous rank
        Cl, Zl = C[ext_lo:(i + 1) * s], Z[ext_lo:(i + 1) * s]
        blocks = [b for b in range(n // m) if lo <= (b + 1) * m - 1 < (i + 1) * s]       # blocks that END in this rank's range (at most s/m + 1 of them)
        ents = []
        for b in blocks:
            a_, e_ = b * m - ext_lo, (b + 1) * m - ext_lo; assert a_ >= 0                 # everything the block needs is local or in the received tail
            ents.append(hca_compress(Cl[a_:e_], Zl[a_:e_], Bp, m)[0])
        pad = s // m + 1 - len(ents); assert pad >= 0
        out.append((torch.stack(ents) if ents else torch.zeros(0, C.shape[1]), pad))
    # stage 2: all-gather then select-and-pad (valid entries first in rank order, padding at the tail)
    return torch.cat([o for o, _ in out], 0), [p for _, p in out]
torch.manual_seed(0); c_ = 8
for (n, R, m) in [(256, 4, 8), (240, 4, 8), (250, 5, 7), (192, 3, 16)]:
    C, Z = torch.randn(n, c_), torch.randn(n, c_); Bp = torch.randn(m, c_) * .1
    ref = hca_compress(C, Z, Bp, m); got, pads = cp_hca(C, Z, Bp, m, R)
    md_ = (ref - got).abs().max().item() if got.shape == ref.shape else float("nan")
    print(f"n={n:4d} ranks={R} m={m:2d} (s={n // R}, s%m={(n // R) % m}):  single-device {tuple(ref.shape)}  vs context-parallel {tuple(got.shape)};  per-rank padding {pads};  max |diff| {md_:.1e}")
    check(f"CP compression (HCA) equals single-device (to 1e-6) for n={n}, R={R}, m={m}, including straddling blocks", got.shape == ref.shape and md_ < 1e-6, f"{md_:.1e}")

n= 256 ranks=4 m= 8 (s=64, s%m=0):  single-device (32, 8)  vs context-parallel (32, 8);  per-rank padding [1, 1, 1, 1];  max |diff| 1.2e-07
PASS CP compression (HCA) equals single-device (to 1e-6) for n=256, R=4, m=8, including straddling blocks   [1.2e-07]
n= 240 ranks=4 m= 8 (s=60, s%m=4):  single-device (30, 8)  vs context-parallel (30, 8);  per-rank padding [1, 0, 1, 0];  max |diff| 1.2e-07
PASS CP compression (HCA) equals single-device (to 1e-6) for n=240, R=4, m=8, including straddling blocks   [1.2e-07]
n= 250 ranks=5 m= 7 (s=50, s%m=1):  single-device (35, 8)  vs context-parallel (35, 8);  per-rank padding [1, 1, 1, 1, 1];  max |diff| 1.2e-07
PASS CP compression (HCA) equals single-device (to 1e-6) for n=250, R=5, m=7, including straddling blocks   [1.2e-07]
n= 192 ranks=3 m=16 (s=64, s%m=0):  single-device (12, 8)  vs context-parallel (12, 8);  per-rank padding [1, 1, 1];  max |diff| 6.0e-08
PASS CP compression (HCA) equals single-device (to 1e-6) for n=192, R=3, m=16, includi

Every case, including those where the block size does not divide the per-rank length (`s % m ≠ 0`, so blocks straddle rank boundaries), reproduces the single-device result to within float32 rounding (about `1e-7`; computing a block's softmax alone versus in a batch rounds differently, which is the very phenomenon of Section 2), and each rank produces at most `s/m + 1` entries, which is why the report fixes the local output length at `s/m + 1` and pads. (Trailing tokens that do not fill a block are dropped, as in the report.)

For **CSA** the compressor is the overlapped one (notebook 1): entry `i` uses block `i` of the `a` series and block `i−1` of the `b` series. We ask the natural question: is "the last `m` entries" enough there? A rank owns the blocks that end in its range; the first such block `j` may straddle the boundary, and its `b`-series half needs block `j−1`, which can reach up to `2m−1` tokens before the boundary.

In [10]:
def csa_compress(Ca, Cb, Za, Zb, Ba, Bb, m):
    nb = Ca.shape[0] // m; c_ = Ca.shape[1]
    Ca_b, Cb_b = Ca[:nb * m].view(nb, m, c_), Cb[:nb * m].view(nb, m, c_); Za_b, Zb_b = Za[:nb * m].view(nb, m, c_) + Ba, Zb[:nb * m].view(nb, m, c_) + Bb
    pC = torch.cat([torch.zeros(1, m, c_), Cb_b[:-1]], 0); pZ = torch.cat([torch.full((1, m, c_), float("-inf")), Zb_b[:-1]], 0)
    S = torch.softmax(torch.cat([Za_b, pZ], 1), 1); return (S[:, :m] * Ca_b).sum(1) + (S[:, m:] * pC).sum(1)
def cp_csa(Ca, Cb, Za, Zb, Ba, Bb, m, R, tail):
    n = Ca.shape[0]; s = n // R; out = []; ok = True
    for i in range(R):
        lo = i * s; ext_lo = max(0, lo - tail)                                           # tail = how many uncompressed entries the previous rank sends
        blocks = [b for b in range(n // m) if lo <= (b + 1) * m - 1 < (i + 1) * s]
        for b in blocks:
            need_lo = max(0, (b - 1) * m)                                                # block b needs tokens from (b-1)m (its b-series half) to (b+1)m-1
            if need_lo < ext_lo: ok = False; continue                                    # not available on this rank
            sl = slice(need_lo, (b + 1) * m); k_ = b - 1 if b > 0 else 0
            sub = csa_compress(Ca[sl], Cb[sl], Za[sl], Zb[sl], Ba, Bb, m)               # compress this window locally; the entry for block b is the last one
            out.append(sub[-1] if b > 0 else sub[0])
    return (torch.stack(out) if ok else None), ok
torch.manual_seed(1); res = {}
for (n, R, m) in [(256, 4, 8), (240, 4, 8), (250, 5, 7)]:
    Ca, Cb, Za, Zb = (torch.randn(n, 8) for _ in range(4)); Ba, Bb = torch.randn(m, 8) * .1, torch.randn(m, 8) * .1; ref = csa_compress(Ca, Cb, Za, Zb, Ba, Bb, m)
    for tail in (m, 2 * m):
        got, ok = cp_csa(Ca, Cb, Za, Zb, Ba, Bb, m, R, tail); res[(n, R, m, tail)] = (ok, ok and bool((got - ref).abs().max() < 1e-6)); print(f"n={n:4d} ranks={R} m={m}  tail sent = {tail:2d} entries ({tail // m}m):  all needed tokens available: {ok};  equal to single device (1e-6): {res[(n, R, m, tail)][1]}")
check("for the overlapped (CSA) compressor, sending only m entries is NOT always enough when blocks straddle a boundary (our formulation)", any(not res[k][0] for k in res if k[3] == k[2]))
check("sending 2m entries always suffices and reproduces the single-device result (to 1e-6)", all(res[k][1] for k in res if k[3] == 2 * k[2]))

n= 256 ranks=4 m=8  tail sent =  8 entries (1m):  all needed tokens available: True;  equal to single device (1e-6): True
n= 256 ranks=4 m=8  tail sent = 16 entries (2m):  all needed tokens available: True;  equal to single device (1e-6): True
n= 240 ranks=4 m=8  tail sent =  8 entries (1m):  all needed tokens available: False;  equal to single device (1e-6): False
n= 240 ranks=4 m=8  tail sent = 16 entries (2m):  all needed tokens available: True;  equal to single device (1e-6): True
n= 250 ranks=5 m=7  tail sent =  7 entries (1m):  all needed tokens available: False;  equal to single device (1e-6): False
n= 250 ranks=5 m=7  tail sent = 14 entries (2m):  all needed tokens available: True;  equal to single device (1e-6): True
PASS for the overlapped (CSA) compressor, sending only m entries is NOT always enough when blocks straddle a boundary (our formulation)
PASS sending 2m entries always suffices and reproduces the single-device result (to 1e-6)


**What we found.** For HCA, the report's scheme works exactly as described. For CSA's overlapped compressor, in our formulation, the previous rank has to send up to `2m − 1` tokens' worth of the `b`-series (and `a`-series where needed), not `m`, whenever the boundary falls in the middle of a block (when the per-rank length is a multiple of `m`, in our runs `m` entries did suffice; with `s % m ≠ 0` they did not). We do not conclude the report is wrong: its description is a summary ("for simplicity ... may omit some tiny details", §2.3.3 says of the attention section, and the context-parallel text does not separate CSA from HCA), and the authors may send both the raw entries and already-projected per-token series, or align boundaries to blocks. What we can say is that the `m`-entry handoff alone does not reproduce our overlapped compressor when the block size does not divide the rank length, and that this is a detail to check in the open-source implementation.

## Closing thoughts

The three pieces share a structure: a distributed or batched computation must reproduce a canonical single-device one. Our checks come out as follows: floating-point addition order, not accuracy, is what makes outputs batch-dependent (a batch-size heuristic changes the bits of most tokens, while a fixed chunking with a pinned merge order, under which partials may be computed in any order, restores them); atomic-style accumulation differs run to run while per-worker buffers plus a fixed global sum do not; whole-matrix knapsack assignment keeps padding under 10 % where element-wise ZeRO would cut matrices, and the MoE flattening wastes under half a percent; BF16 is adequate for Newton-Schulz and, with FP32 local sums, for gradient exchange; and the two-stage context-parallel compression reproduces single-device compression to float32 rounding for HCA.

**What is open, and what we did not test.** None of the overhead claims (negligible for batch invariance, under 10 % memory for the buckets, the throughput effects) can be measured on a CPU. Our weight-shape list is a reconstruction. And the report does not say how its context-parallel handoff treats the overlapped compressor.

## The series

This is the last of nine notebooks on the DeepSeek-V4 report: (1) hybrid attention, (2) mHC in V4 (with the standalone [mHC notebook](../models/mhc/mhc_from_scratch.ipynb)), (3) Muon, (4) MoE changes, (5) training-stability techniques, (6) post-training with OPD, (7) FP4 QAT, (8) MegaMoE overlap, (9) this one. Across them the pattern we most often found is that a stated property holds exactly when we test its mechanism in isolation (causality, the doubly stochastic constraint, the FP4→FP8 threshold, bitwise reproducibility), while the claims about *scale* (quality, speed, overhead, spike prevention) are the ones a toy cannot reach, and we have marked them as such.

Credits: everything about V4 is DeepSeek-AI's; flash-decoding is Dao et al.; ZeRO is Rajbhandari et al.; the OCP microscaling and stochastic-rounding ideas are from the literature the report cites.

In [11]:
print(f"{'#':>2s}  {'status':6s}  check")
for i, (n_, ok, d_) in enumerate(CHECKS, 1): print(f"{i:2d}  {'PASS' if ok else 'FAIL':6s}  {n_}")
failed = [n_ for n_, ok, _ in CHECKS if not ok]; print(f"\n{len(CHECKS) - len(failed)} passed, {len(failed)} failed")
assert not failed, failed

 #  status  check
 1  PASS    float32 addition is not associative: three groupings of (1e8, 1, -1e8) give both 0 and 1
 2  PASS    a batch-size-dependent split makes the same token's output depend on its batch (most tokens differ in float32 bits)
 3  PASS    the differences are rounding-level (< 1e-5): the kernels agree mathematically
 4  PASS    both split choices are within 1e-5 of the float64 reference (the problem is reproducibility, not accuracy)
 5  PASS    kernel 2 (partials in any order, merge in the fixed order) is bit-identical to kernel 1 for all tokens
 6  PASS    a different merge grouping (pairwise tree) is NOT bit-identical for most tokens - the order must be pinned
 7  PASS    with a fixed chunk size the output of a token does not depend on the batch it is in
 8  PASS    atomic-style accumulation in random arrival order gives different float32 bits from run to run
 9  PASS    per-worker buffers plus a fixed-order global sum are bit-identical every run
10  PASS    both a